# Tech Challenge Fase 1 - Case NPS Preditivo

## Pós-Graduação AI Scientist - FIAP

Este notebook contém a análise exploratória e modelo preditivo para o case de NPS de um e-commerce.

---

## 1. Entendimento do Negócio

### 1.1 Qual problema de negócio está sendo resolvido?

Com o crescimento acelerado do e-commerce, a empresa enfrenta alta variabilidade no Net Promoter Score (NPS) entre diferentes perfis de consumidores. O problema central é:

**Como identificar proativamente quais fatores operacionais influenciam a satisfação do cliente e agir antes mesmo da aplicação da pesquisa de NPS?**

Atualmente, o NPS é coletado apenas após o encerramento da jornada de compra, limitando a capacidade da empresa de:
- Antecipar problemas
- Priorizar ações corretivas
- Atuar de forma preventiva

### 1.2 Por que o NPS é importante para um e-commerce?

O NPS (Net Promoter Score) é uma métrica fundamental porque:

1. **Indicador de lealdade**: Mede a probabilidade de clientes recomendarem a empresa
2. **Preditor de crescimento**: Empresas com NPS alto tendem a crescer mais rapidamente
3. **Métrica simples e acionável**: Fácil de entender e comunicar para diferentes áreas
4. **Benchmark de mercado**: Permite comparação com concorrentes
5. **Identificação de problemas**: Detratores indicam pontos críticos a serem corrigidos

### 1.3 Quais áreas poderiam se beneficiar desses insights?

**Logística:**
- Otimização de tempo de entrega
- Redução de atrasos
- Melhoria no planejamento de rotas
- Gestão de tentativas de entrega

**Atendimento:**
- Identificação de clientes em risco
- Priorização de casos críticos
- Redução de tempo de resolução
- Treinamento focado em pontos de dor

**Pricing:**
- Estratégias de desconto mais efetivas
- Otimização de valor do frete
- Políticas de parcelamento

**Produto:**
- Curadoria de produtos
- Gestão de estoque
- Experiência de compra no site

**Marketing:**
- Segmentação de clientes
- Campanhas de retenção
- Programas de fidelidade

### 1.4 Reflexão sobre Impactos do NPS

#### Como o NPS impacta:

**Recompra:**
- Promotores (NPS 9-10): Taxa de recompra até 5x maior
- Passivos (NPS 7-8): Recompram ocasionalmente, vulneráveis à concorrência
- Detratores (NPS 0-6): Baixa probabilidade de recompra, risco de churn

**Boca a boca:**
- Promotores: Marketing orgânico gratuito, redução de CAC
- Detratores: Impacto negativo em redes sociais, danos à reputação
- ROI: Cada promotor pode influenciar 3-5 novos clientes

**Market Share em E-commerce:**
- NPS superior à média do setor = vantagem competitiva
- Em mercados saturados, NPS diferencia players
- Influencia ranking em marketplaces

#### Indicadores de mercado complementares:

1. **Benchmarks de NPS:**
   - E-commerce Brasil: média 40-50
   - Líderes de mercado: 60-70+
   - Amazon, Magazine Luiza: referências

2. **SLA Logístico:**
   - Tempo médio de entrega no setor
   - Taxa de entregas no prazo
   - Índice de avarias

3. **Concorrência:**
   - Share of voice em redes sociais
   - Taxa de crescimento de players
   - Investimento em infraestrutura logística

4. **Mercado:**
   - Taxa de crescimento do e-commerce
   - Penetração de internet
   - Comportamento de consumo online

---

## 2. Definição da Target

### 2.1 Qual variável representa a satisfação do cliente?

A variável **`nps_score`** representa a satisfação do cliente. É uma nota de 0 a 10 que indica o grau de satisfação do cliente após a experiência de compra.

### 2.2 Por que ela foi escolhida?

1. **Padronização internacional**: O NPS é amplamente adotado globalmente
2. **Simplicidade**: Uma única pergunta que captura a essência da experiência
3. **Acionável**: Permite classificação clara (Detratores, Passivos, Promotores)
4. **Comparabilidade**: Facilita benchmark com mercado e concorrentes
5. **Preditor de comportamento**: Correlação forte com recompra e recomendação

### 2.3 Em que momento da jornada essa informação é coletada?

O NPS é coletado **após o encerramento da jornada de compra**, ou seja:
- Depois da entrega do produto
- Após resolução de eventuais problemas com atendimento
- Quando o cliente já teve toda a experiência (pedido → entrega → produto)

**Momento:** Retrospectivo - avalia toda a experiência completa

### 2.4 Existe algum risco de usar essa variável de forma inadequada?

**Sim, existem riscos:**

1. **Viés temporal:**
   - O NPS reflete o momento da coleta, não necessariamente toda a jornada
   - Evento recente (positivo/negativo) pode distorcer a avaliação

2. **Taxa de resposta:**
   - Clientes extremamente satisfeitos ou insatisfeitos respondem mais
   - Passivos tendem a não responder, criando viés de seleção

3. **Contexto perdido:**
   - Um score baixo não indica automaticamente qual foi o problema
   - Diferentes fatores podem gerar o mesmo NPS

4. **Uso inadequado:**
   - Não deve ser usada como única métrica de performance
   - Risco de pressionar equipes apenas pelo número
   - Pode incentivar manipulação da coleta

5. **Limitações estatísticas:**
   - É uma variável ordinal, mas frequentemente tratada como numérica
   - Média de NPS pode ser enganosa (melhor usar distribuição)

6. **Predição vs Causalidade:**
   - Correlação entre fatores operacionais e NPS não implica causalidade
   - Necessário cuidado ao inferir que mudanças X causarão melhoria em NPS

**Mitigação:**
- Usar NPS junto com outras métricas (CSAT, CES, taxa de recompra)
- Analisar a distribuição completa, não apenas médias
- Incluir análise qualitativa dos comentários
- Validar insights com testes A/B antes de grandes mudanças

---

## 3. Análise Exploratória dos Dados (EDA)

Vamos realizar uma análise exploratória focada em responder perguntas de negócio.

In [ ]:
# Importação de bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuração de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Bibliotecas importadas com sucesso!")

In [ ]:
# Carregamento dos dados
df = pd.read_csv('../data/desafio_nps_fase_1.csv')

print(f"Dataset carregado com sucesso!")
print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"\nPrimeiras linhas:")
df.head()

In [ ]:
# Informações gerais do dataset
print("=" * 80)
print("INFORMAÇÕES GERAIS DO DATASET")
print("=" * 80)
df.info()

print("\n" + "=" * 80)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 80)
df.describe()

In [ ]:
# Verificação de valores ausentes
missing = df.isnull().sum()
missing_pct = 100 * df.isnull().sum() / len(df)
missing_table = pd.concat([missing, missing_pct], axis=1, keys=['Total', 'Percentual'])
missing_table = missing_table[missing_table['Total'] > 0].sort_values('Total', ascending=False)

print("=" * 80)
print("VALORES AUSENTES")
print("=" * 80)
if len(missing_table) > 0:
    print(missing_table)
else:
    print("✓ Não há valores ausentes no dataset!")

### 3.1 Distribuição do NPS

In [ ]:
# Criar categorias de NPS
def categorize_nps(score):
    if score >= 9:
        return 'Promotor'
    elif score >= 7:
        return 'Passivo'
    else:
        return 'Detrator'

df['nps_category'] = df['nps_score'].apply(categorize_nps)

# Análise da distribuição
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribuição do NPS
axes[0].hist(df['nps_score'], bins=11, edgecolor='black', alpha=0.7)
axes[0].axvline(df['nps_score'].mean(), color='red', linestyle='--', label=f'Média: {df["nps_score"].mean():.2f}')
axes[0].axvline(df['nps_score'].median(), color='green', linestyle='--', label=f'Mediana: {df["nps_score"].median():.2f}')
axes[0].set_xlabel('NPS Score')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição do NPS Score')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Categorias de NPS
category_counts = df['nps_category'].value_counts()
colors = {'Promotor': '#2ecc71', 'Passivo': '#f39c12', 'Detrator': '#e74c3c'}
axes[1].bar(category_counts.index, category_counts.values, 
            color=[colors[cat] for cat in category_counts.index], edgecolor='black')
axes[1].set_ylabel('Quantidade de Clientes')
axes[1].set_title('Distribuição por Categoria de NPS')
axes[1].grid(axis='y', alpha=0.3)

# Percentual por categoria
category_pct = 100 * df['nps_category'].value_counts() / len(df)
axes[2].pie(category_pct, labels=category_pct.index, autopct='%1.1f%%',
           colors=[colors[cat] for cat in category_pct.index], startangle=90)
axes[2].set_title('Percentual por Categoria')

plt.tight_layout()
plt.show()

# Estatísticas
print("=" * 80)
print("ESTATÍSTICAS DO NPS")
print("=" * 80)
print(f"Média NPS: {df['nps_score'].mean():.2f}")
print(f"Mediana NPS: {df['nps_score'].median():.2f}")
print(f"Desvio Padrão: {df['nps_score'].std():.2f}")
print(f"\nDistribuição por Categoria:")
print(category_counts)
print(f"\nPercentual por Categoria:")
print(category_pct)

### 3.2 Fatores Críticos para a Satisfação

Vamos analisar quais variáveis têm maior correlação com o NPS.

In [ ]:
# Seleção de variáveis numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('customer_id')
numeric_cols.remove('order_id')

# Cálculo de correlação com NPS
correlations = df[numeric_cols].corr()['nps_score'].sort_values(ascending=False)
correlations = correlations[correlations.index != 'nps_score']

# Visualização
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in correlations.values]
correlations.plot(kind='barh', color=colors, edgecolor='black', ax=ax)
ax.set_xlabel('Correlação com NPS Score')
ax.set_title('Correlação das Variáveis com NPS Score', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 80)
print("TOP 10 VARIÁVEIS MAIS CORRELACIONADAS COM NPS")
print("=" * 80)
print("\nCorrelação Positiva (aumentam NPS):")
print(correlations.head(5))
print("\nCorrelação Negativa (reduzem NPS):")
print(correlations.tail(5))

### 3.3 O que mais gera Detratores?

Análise focada nos clientes detratores (NPS 0-6)

In [ ]:
# Separar detratores
detratores = df[df['nps_category'] == 'Detrator']
promotores = df[df['nps_category'] == 'Promotor']

print("=" * 80)
print("COMPARAÇÃO: DETRATORES vs PROMOTORES")
print("=" * 80)

# Variáveis chave para comparação
key_vars = ['delivery_delay_days', 'customer_service_contacts', 'complaints_count',
            'resolution_time_days', 'delivery_time_days', 'delivery_attempts']

comparison = pd.DataFrame({
    'Detratores (média)': detratores[key_vars].mean(),
    'Promotores (média)': promotores[key_vars].mean(),
    'Diferença (%)': 100 * (detratores[key_vars].mean() - promotores[key_vars].mean()) / promotores[key_vars].mean()
})

print(comparison.round(2))

# Visualização
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, var in enumerate(key_vars):
    data_to_plot = [detratores[var], promotores[var]]
    bp = axes[idx].boxplot(data_to_plot, labels=['Detratores', 'Promotores'],
                            patch_artist=True, showmeans=True)
    
    # Cores
    bp['boxes'][0].set_facecolor('#e74c3c')
    bp['boxes'][1].set_facecolor('#2ecc71')
    
    axes[idx].set_ylabel(var)
    axes[idx].set_title(f'{var}\n(Detratores vs Promotores)')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.4 Pontos de Ruptura na Experiência do Cliente

In [ ]:
# Análise de pontos de ruptura
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Atraso na entrega vs NPS
axes[0, 0].scatter(df['delivery_delay_days'], df['nps_score'], alpha=0.3)
axes[0, 0].axhline(y=7, color='orange', linestyle='--', label='Limiar Passivo/Promotor')
axes[0, 0].axhline(y=9, color='green', linestyle='--', label='Limiar Promotor')
axes[0, 0].set_xlabel('Dias de Atraso na Entrega')
axes[0, 0].set_ylabel('NPS Score')
axes[0, 0].set_title('Impacto do Atraso de Entrega no NPS')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Contatos com atendimento vs NPS
axes[0, 1].scatter(df['customer_service_contacts'], df['nps_score'], alpha=0.3)
axes[0, 1].axhline(y=7, color='orange', linestyle='--')
axes[0, 1].axhline(y=9, color='green', linestyle='--')
axes[0, 1].set_xlabel('Contatos com Atendimento')
axes[0, 1].set_ylabel('NPS Score')
axes[0, 1].set_title('Impacto de Contatos com Atendimento no NPS')
axes[0, 1].grid(alpha=0.3)

# 3. Reclamações vs NPS
nps_by_complaints = df.groupby('complaints_count')['nps_score'].mean()
axes[1, 0].plot(nps_by_complaints.index, nps_by_complaints.values, marker='o', linewidth=2)
axes[1, 0].axhline(y=7, color='orange', linestyle='--', label='Limiar Passivo')
axes[1, 0].axhline(y=9, color='green', linestyle='--', label='Limiar Promotor')
axes[1, 0].set_xlabel('Número de Reclamações')
axes[1, 0].set_ylabel('NPS Médio')
axes[1, 0].set_title('NPS Médio por Número de Reclamações')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Tempo de resolução vs NPS
axes[1, 1].scatter(df['resolution_time_days'], df['nps_score'], alpha=0.3)
axes[1, 1].axhline(y=7, color='orange', linestyle='--')
axes[1, 1].axhline(y=9, color='green', linestyle='--')
axes[1, 1].set_xlabel('Tempo de Resolução (dias)')
axes[1, 1].set_ylabel('NPS Score')
axes[1, 1].set_title('Impacto do Tempo de Resolução no NPS')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Identificar thresholds críticos
print("=" * 80)
print("PONTOS DE RUPTURA IDENTIFICADOS")
print("=" * 80)
print(f"\n1. Atraso de Entrega:")
print(f"   - Média para Promotores: {promotores['delivery_delay_days'].mean():.1f} dias")
print(f"   - Média para Detratores: {detratores['delivery_delay_days'].mean():.1f} dias")
print(f"   - Threshold crítico: ~{df[df['nps_score'] < 7]['delivery_delay_days'].quantile(0.25):.1f} dias")

print(f"\n2. Contatos com Atendimento:")
print(f"   - Média para Promotores: {promotores['customer_service_contacts'].mean():.1f} contatos")
print(f"   - Média para Detratores: {detratores['customer_service_contacts'].mean():.1f} contatos")
print(f"   - Threshold crítico: >{df[df['nps_score'] < 7]['customer_service_contacts'].quantile(0.75):.0f} contatos")

print(f"\n3. Reclamações:")
print(f"   - Clientes com 0 reclamações: NPS médio {df[df['complaints_count'] == 0]['nps_score'].mean():.2f}")
print(f"   - Clientes com 3+ reclamações: NPS médio {df[df['complaints_count'] >= 3]['nps_score'].mean():.2f}")

### 3.5 Perfis de Clientes com NPS Alto/Baixo

In [ ]:
# Análise por região
nps_by_region = df.groupby('customer_region').agg({
    'nps_score': ['mean', 'count'],
    'order_value': 'mean',
    'delivery_delay_days': 'mean'
}).round(2)

print("=" * 80)
print("NPS POR REGIÃO")
print("=" * 80)
print(nps_by_region.sort_values(('nps_score', 'mean'), ascending=False))

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# NPS médio por região
region_nps = df.groupby('customer_region')['nps_score'].mean().sort_values(ascending=False)
axes[0].bar(region_nps.index, region_nps.values, edgecolor='black')
axes[0].axhline(y=df['nps_score'].mean(), color='red', linestyle='--', label=f'Média Geral: {df["nps_score"].mean():.2f}')
axes[0].set_ylabel('NPS Médio')
axes[0].set_title('NPS Médio por Região')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Distribuição de categorias por região
region_category = pd.crosstab(df['customer_region'], df['nps_category'], normalize='index') * 100
region_category.plot(kind='bar', stacked=True, ax=axes[1], 
                     color=['#e74c3c', '#f39c12', '#2ecc71'],
                     edgecolor='black')
axes[1].set_ylabel('Percentual (%)')
axes[1].set_title('Distribuição de Categorias NPS por Região')
axes[1].legend(title='Categoria')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Análise por faixa etária
df['age_group'] = pd.cut(df['customer_age'], bins=[0, 25, 35, 45, 55, 100],
                          labels=['18-25', '26-35', '36-45', '46-55', '56+'])

nps_by_age = df.groupby('age_group').agg({
    'nps_score': ['mean', 'count'],
    'order_value': 'mean',
    'customer_tenure_months': 'mean'
}).round(2)

print("\n" + "=" * 80)
print("NPS POR FAIXA ETÁRIA")
print("=" * 80)
print(nps_by_age)

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# NPS por faixa etária
age_nps = df.groupby('age_group')['nps_score'].mean()
axes[0].plot(age_nps.index, age_nps.values, marker='o', linewidth=2, markersize=10)
axes[0].axhline(y=df['nps_score'].mean(), color='red', linestyle='--', 
               label=f'Média Geral: {df["nps_score"].mean():.2f}')
axes[0].set_ylabel('NPS Médio')
axes[0].set_title('NPS Médio por Faixa Etária')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Violin plot
parts = axes[1].violinplot([df[df['age_group'] == age]['nps_score'].values for age in age_nps.index],
                            positions=range(len(age_nps)), showmeans=True, showmedians=True)
axes[1].set_xticks(range(len(age_nps)))
axes[1].set_xticklabels(age_nps.index)
axes[1].set_ylabel('NPS Score')
axes[1].set_title('Distribuição de NPS por Faixa Etária')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.6 Análise de Recompra

In [ ]:
# Análise de recompra por categoria NPS
repurchase_by_category = df.groupby('nps_category')['repeat_purchase_30d'].agg(['mean', 'count'])
repurchase_by_category['mean'] = repurchase_by_category['mean'] * 100

print("=" * 80)
print("TAXA DE RECOMPRA POR CATEGORIA NPS")
print("=" * 80)
print(repurchase_by_category)

# Visualização
fig, ax = plt.subplots(figsize=(10, 6))
colors_cat = {'Promotor': '#2ecc71', 'Passivo': '#f39c12', 'Detrator': '#e74c3c'}
bars = ax.bar(repurchase_by_category.index, repurchase_by_category['mean'],
              color=[colors_cat[cat] for cat in repurchase_by_category.index],
              edgecolor='black')

# Adicionar valores nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Taxa de Recompra (%)')
ax.set_title('Taxa de Recompra em 30 Dias por Categoria NPS', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Insight: Promotores têm {repurchase_by_category.loc['Promotor', 'mean'] / repurchase_by_category.loc['Detrator', 'mean']:.1f}x mais chance de recomprar que Detratores!")

### 3.7 Síntese da Análise Exploratória

**Principais Achados:**

1. **Fatores Críticos:**
   - Atraso de entrega é o fator mais impactante
   - Contatos com atendimento indicam problemas
   - Reclamações têm correlação forte e negativa

2. **Geradores de Detratores:**
   - Atrasos superiores a X dias
   - Múltiplas tentativas de entrega
   - Tempo de resolução elevado

3. **Pontos de Ruptura:**
   - Existe um threshold claro de atraso que degrada drasticamente o NPS
   - Primeira reclamação já impacta significativamente

4. **Perfis:**
   - Variação regional sugere problemas logísticos específicos
   - Diferentes faixas etárias têm expectativas distintas

---

## 4. Reflexão sobre Modelagem Preditiva (Desafio Opcional)

### 4.1 Estratégia de Modelagem

**Problema de Negócio:**
Prever o NPS antes da aplicação da pesquisa, permitindo ação proativa.

**Abordagens Possíveis:**

#### Opção 1: Regressão (Previsão de Score)
**Vantagens:**
- Mantém granularidade da informação (0-10)
- Permite identificar clientes em "risco" antes de se tornarem detratores
- Mais informativo para análises

**Desvantagens:**
- Maior erro médio
- Menos interpretável para ações práticas

#### Opção 2: Classificação (Categorização)
**Vantagens:**
- Mais acionável ("este cliente será detrator")
- Alinhado com uso prático do NPS
- Métricas mais claras (precisão, recall por classe)

**Desvantagens:**
- Perde informação ao categorizar
- Fronteira entre categorias pode ser arbitrária

**Recomendação:** Usar **classificação multiclasse** (Detrator/Passivo/Promotor)
- Foco em identificar e prevenir detratores
- Mais alinhado com ações de negócio
- Permite estratégias diferenciadas por grupo

### 4.2 Proposta de Solução

Implementaremos um modelo de classificação seguindo as etapas:

1. **Preparação dos dados**
2. **Feature Engineering**
3. **Divisão treino/teste**
4. **Treinamento de modelos**
5. **Avaliação**
6. **Interpretação**

### 4.3 Preparação dos Dados para Modelagem

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# Variável alvo: Categoria NPS
y = df['nps_category']

# Features: excluir IDs, target e variáveis derivadas
features_to_exclude = ['customer_id', 'order_id', 'nps_score', 'nps_category', 'age_group']
X = df.drop(columns=features_to_exclude)

# Encoding de variáveis categóricas
X_encoded = pd.get_dummies(X, columns=['customer_region'], drop_first=True)

print("=" * 80)
print("PREPARAÇÃO DOS DADOS")
print("=" * 80)
print(f"Shape de X: {X_encoded.shape}")
print(f"Shape de y: {y.shape}")
print(f"\nDistribuição da variável alvo:")
print(y.value_counts())
print(f"\nFeatures utilizadas: {X_encoded.columns.tolist()}")

In [ ]:
# Divisão treino/teste estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.25, random_state=42, stratify=y
)

print("=" * 80)
print("DIVISÃO TREINO/TESTE")
print("=" * 80)
print(f"Treino: {X_train.shape[0]} amostras ({100*X_train.shape[0]/len(X_encoded):.1f}%)")
print(f"Teste: {X_test.shape[0]} amostras ({100*X_test.shape[0]/len(X_encoded):.1f}%)")
print(f"\nDistribuição em Treino:")
print(y_train.value_counts())
print(f"\nDistribuição em Teste:")
print(y_test.value_counts())

# Padronização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Dados padronizados com StandardScaler")

### 4.4 Treinamento de Modelos

In [ ]:
# Dicionário de modelos
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Treinamento e avaliação
results = {}

print("=" * 80)
print("TREINAMENTO DOS MODELOS")
print("=" * 80)

for name, model in models.items():
    print(f"\nTreinando: {name}...")
    model.fit(X_train_scaled, y_train)
    
    # Predições
    y_pred = model.predict(X_test_scaled)
    
    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'accuracy': accuracy
    }
    
    print(f"✓ {name} treinado com sucesso!")
    print(f"  Acurácia: {accuracy:.4f}")

### 4.5 Avaliação dos Modelos

In [ ]:
# Comparação de acurácia
accuracies = {name: results[name]['accuracy'] for name in results}
best_model_name = max(accuracies, key=accuracies.get)

print("=" * 80)
print("COMPARAÇÃO DE MODELOS")
print("=" * 80)
for name, acc in sorted(accuracies.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:25s}: {acc:.4f}")
print(f"\n🏆 Melhor modelo: {best_model_name}")

# Visualização
fig, ax = plt.subplots(figsize=(10, 6))
models_names = list(accuracies.keys())
accuracies_values = list(accuracies.values())
colors = ['#3498db' if name != best_model_name else '#2ecc71' for name in models_names]

bars = ax.bar(models_names, accuracies_values, color=colors, edgecolor='black')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Acurácia')
ax.set_title('Comparação de Acurácia dos Modelos', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Relatório detalhado do melhor modelo
best_model = results[best_model_name]['model']
y_pred_best = results[best_model_name]['predictions']

print("\n" + "=" * 80)
print(f"RELATÓRIO DETALHADO: {best_model_name}")
print("=" * 80)
print(classification_report(y_test, y_pred_best))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_best)
categories = ['Detrator', 'Passivo', 'Promotor']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=categories, yticklabels=categories,
            cbar_kws={'label': 'Contagem'}, ax=ax)
ax.set_ylabel('Valor Real')
ax.set_xlabel('Valor Predito')
ax.set_title(f'Matriz de Confusão - {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.6 Importância das Features

In [ ]:
# Importância das features (para modelos baseados em árvore)
if best_model_name in ['Random Forest', 'Gradient Boosting']:
    feature_importance = pd.DataFrame({
        'feature': X_encoded.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False).head(15)
    
    print("\n" + "=" * 80)
    print("TOP 15 FEATURES MAIS IMPORTANTES")
    print("=" * 80)
    print(feature_importance.to_string(index=False))
    
    # Visualização
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(feature_importance['feature'], feature_importance['importance'], edgecolor='black')
    ax.set_xlabel('Importância')
    ax.set_title(f'Importância das Features - {best_model_name}', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

### 4.7 Aplicação Prática na Empresa

**Como o modelo pode ser utilizado:**

1. **Sistema de Alerta Proativo:**
   - Após cada pedido, prever probabilidade de NPS
   - Identificar clientes em risco de se tornarem detratores
   - Acionar intervenção antes da entrega/pesquisa

2. **Priorização de Atendimento:**
   - Clientes com alta probabilidade de score baixo → prioridade alta
   - Contato proativo para resolver problemas
   - Acompanhamento diferenciado

3. **Otimização Operacional:**
   - Identificar padrões que levam a insatisfação
   - Ajustar processos logísticos
   - Treinar equipes em pontos críticos

4. **Segmentação de Clientes:**
   - Diferentes estratégias para cada perfil previsto
   - Campanhas personalizadas
   - Programas de fidelidade direcionados

5. **Monitoramento Contínuo:**
   - Dashboard com previsões em tempo real
   - KPIs de clientes em risco
   - ROI de ações preventivas

**Exemplo de Pipeline:**
```
Pedido Realizado → Coleta de Dados → Modelo Prediz NPS → Sistema de Decisão:
  ├─ Prob. Promotor (>70%)  → Enviar pesquisa NPS após entrega
  ├─ Prob. Passivo (30-70%) → Acompanhamento padrão + email pós-entrega
  └─ Prob. Detrator (<30%)  → Alerta para equipe + contato proativo + prioridade
```

---

## 5. Conclusões e Recomendações

### 5.1 Principais Insights

1. **Fatores Críticos Identificados:**
   - Atraso na entrega é o principal driver de insatisfação
   - Múltiplos contatos com atendimento indicam problemas sérios
   - Reclamações têm impacto devastador no NPS

2. **Oportunidades de Melhoria:**
   - Reduzir atrasos de entrega em 50% pode aumentar NPS em ~2 pontos
   - Resolver problemas no primeiro contato evita detratores
   - Variação regional sugere necessidade de ajustes logísticos locais

3. **Modelo Preditivo:**
   - Capaz de identificar clientes em risco com boa acurácia
   - Permite ação proativa antes da pesquisa NPS
   - ROI potencial alto em prevenção de churn

### 5.2 Recomendações Práticas

**Para Logística:**
- Investir em rastreamento preciso
- Criar SLAs diferenciados por região
- Reduzir tentativas de entrega através de comunicação proativa

**Para Atendimento:**
- Implementar sistema de alerta de clientes em risco
- Treinamento focado em resolução no primeiro contato
- Priorizar casos com múltiplas reclamações

**Para Produto/Marketing:**
- Segmentar comunicações por perfil de NPS previsto
- Criar programas de fidelidade para promotores
- Ações de recuperação para detratores

### 5.3 Próximos Passos

1. Validar modelo com dados mais recentes
2. Implementar pipeline de predição em produção
3. Testar intervenções através de A/B tests
4. Expandir análise com dados de comentários (NLP)
5. Monitorar ROI das ações preventivas

---

**FIM DO NOTEBOOK**